# 🏨 Modelos No Supervisados con Datos de Hotel
## Segmentación de huéspedes · Detección de anomalías · Reducción de dimensionalidad

---

En esta actividad trabajarás con un dataset de **500 reservas de hotel** y aplicarás tres técnicas de aprendizaje **no supervisado** en Python:

| Técnica | Algoritmo | ¿Para qué? |
|---|---|---|
| 🔵 Segmentación | K-Means | Agrupar huéspedes por perfil de comportamiento |
| 🟣 Visualización | PCA | Ver patrones ocultos reduciendo dimensiones |
| 🟤 Clustering jerárquico | AgglomerativeClustering | Explorar la estructura del dato sin definir K |
| 🔴 Detección de anomalías | DBSCAN | Encontrar reservas atípicas o sospechosas |

> **En el aprendizaje no supervisado NO hay respuesta correcta.** El modelo descubre estructura a partir de los propios datos, sin etiquetas.

**⏱ Duración:** 50 minutos | **🐍 Lenguaje:** Python | **☁️ Entorno:** Google Colab

> Ejecuta cada celda en orden con `Shift + Enter`. Lee los bloques de texto antes de ejecutar.

---
## ⚙️ Instalación de dependencias
Solo necesaria la primera vez que abras el notebook en Colab.

In [ ]:
# Colab ya incluye la mayoría de estas librerías, pero forzamos versiones estables
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy
print('✅ Librerías listas')

---
## PARTE 1 · Generar y explorar el dataset 🏨
**⏱ 5 minutos**

Generamos un dataset de **500 reservas de hotel** con características realistas:
tipo de hotel, canal de reserva, precio, perfil del huésped, duración de la estancia y más.

No importaremos ningún CSV externo — el dataset se construye aquí mismo para que puedas modificarlo y experimentar.

In [ ]:
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

tipos_hotel      = ['Resort', 'City Hotel']
meses            = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
                    'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']
paises           = ['España','Francia','Portugal','Alemania','Italia','Reino Unido','USA','Brasil']
canales          = ['Directo','OTA','Agencia','Corporativo']
tipos_habitacion = ['Individual','Doble','Suite','Familiar']

rows = []
for i in range(500):
    tipo         = random.choice(tipos_hotel)
    mes          = random.choice(meses)
    noches       = random.randint(1, 14)
    adultos      = random.randint(1, 4)
    ninos        = random.randint(0, 2)
    pais         = random.choice(paises)
    canal        = random.choice(canales)
    habitacion   = random.choice(tipos_habitacion)
    precio_noche = round(random.uniform(50, 400), 2)
    precio_total = round(precio_noche * noches, 2)
    solicitudes  = random.randint(0, 5)
    prev_reservas= random.randint(0, 10)
    anticipacion = random.randint(0, 365)
    estrellas    = random.choice([3, 4, 5])
    valoracion   = round(random.uniform(5.0, 10.0), 1)

    # Añadimos ~15 reservas con comportamiento extremo (anomalías)
    if i < 15:
        precio_noche = round(random.uniform(800, 1500), 2)
        noches       = random.randint(20, 40)
        precio_total = round(precio_noche * noches, 2)
        anticipacion = random.randint(0, 5)

    rows.append([i+1, tipo, mes, noches, adultos, ninos, pais, canal,
                 habitacion, precio_noche, precio_total,
                 solicitudes, prev_reservas, anticipacion,
                 estrellas, valoracion])

columnas = ['id','tipo_hotel','mes_llegada','noches','adultos','ninos',
            'pais_origen','canal_reserva','tipo_habitacion','precio_noche',
            'precio_total','solicitudes_especiales','reservas_previas',
            'dias_anticipacion','estrellas_hotel','valoracion_cliente']

df = pd.DataFrame(rows, columns=columnas)
print(f'✅ Dataset generado: {df.shape[0]} filas · {df.shape[1]} columnas')
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print('=== ESTADÍSTICAS NUMÉRICAS ===')
print(df.describe().round(2))

# Distribuciones de las columnas clave
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
cols_num = ['noches','precio_noche','precio_total',
            'dias_anticipacion','solicitudes_especiales','valoracion_cliente']

for ax, col in zip(axes.flat, cols_num):
    ax.hist(df[col], bins=25, color='#5C6BC0', edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribución de variables numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 💬 Reflexión 1
> **¿Notas algo inusual en las distribuciones de `precio_noche` o `noches`? ¿Por qué hay valores tan altos comparados con la mayoría?**
>
> *(Escribe tu respuesta aquí haciendo doble clic en esta celda)*

---

---
## PARTE 2 · Preprocesamiento y reducción de dimensionalidad (PCA) 🟣
**⏱ 8 minutos**

Antes de aplicar cualquier algoritmo de clustering debemos:
1. **Seleccionar las features numéricas** relevantes
2. **Normalizar** con `StandardScaler` (K-Means es sensible a la escala)
3. **Reducir a 2D con PCA** para poder visualizar los datos

### ¿Qué es PCA?
**Principal Component Analysis** transforma las variables originales en nuevos ejes (componentes principales) que capturan la mayor varianza posible. Con PCA podemos ver en 2D un dataset que tiene 8 dimensiones.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

features_cluster = ['noches', 'adultos', 'ninos', 'precio_noche',
                    'precio_total', 'solicitudes_especiales',
                    'reservas_previas', 'dias_anticipacion',
                    'estrellas_hotel', 'valoracion_cliente']

X = df[features_cluster].copy()

# Normalización
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'✅ Datos normalizados: {X_scaled.shape[0]} reservas · {X_scaled.shape[1]} features')
print('\nMedia tras escalar (debe ser ~0):', X_scaled.mean(axis=0).round(3))
print('Std  tras escalar (debe ser ~1) :', X_scaled.std(axis=0).round(3))

In [ ]:
# PCA para visualización
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

varianza = pca.explained_variance_ratio_
print(f'Varianza explicada por PC1: {varianza[0]:.2%}')
print(f'Varianza explicada por PC2: {varianza[1]:.2%}')
print(f'Varianza total capturada  : {sum(varianza):.2%}')

# Visualizar en 2D
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                alpha=0.5, s=40, c='#5C6BC0', edgecolors='white', linewidth=0.3)
axes[0].set_xlabel(f'PC1 ({varianza[0]:.1%} varianza)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({varianza[1]:.1%} varianza)', fontsize=11)
axes[0].set_title('Datos proyectados en 2D (PCA)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Varianza acumulada con más componentes
pca_full = PCA(random_state=42).fit(X_scaled)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(var_acum)+1), var_acum, 'o-', color='#7B1FA2', linewidth=2, markersize=7)
axes[1].axhline(y=0.90, color='gray', linestyle='--', alpha=0.7, label='90% varianza')
axes[1].fill_between(range(1, len(var_acum)+1), var_acum, alpha=0.15, color='#7B1FA2')
axes[1].set_xlabel('Número de componentes', fontsize=11)
axes[1].set_ylabel('Varianza explicada acumulada', fontsize=11)
axes[1].set_title('¿Cuántos componentes necesitamos?', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 💬 Reflexión 2
> 1. **¿Con cuántos componentes principales capturas el 90% de la varianza del dataset?**
> 2. **¿Notas alguna agrupación visual en el gráfico PCA 2D? ¿Ves puntos alejados del resto?**
>
> *(Escribe tu respuesta aquí)*

---

---
## PARTE 3 · K-Means: Segmentación de huéspedes 🔵
**⏱ 12 minutos**

**K-Means** divide los datos en K grupos buscando minimizar la distancia de cada punto a su centroide.

El reto: **¿cuántos clusters usar?** Usaremos dos métodos para decidirlo:
- **Método del codo:** buscamos donde la inercia deja de bajar rápido
- **Silhouette Score:** mide la calidad de la separación entre clusters (más alto = mejor)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inercias    = []
silhouettes = []
rango_k     = range(2, 9)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del codo
axes[0].plot(rango_k, inercias, 'o-', color='#1565C0', linewidth=2, markersize=9)
axes[0].fill_between(rango_k, inercias, alpha=0.08, color='#1565C0')
axes[0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inercia (suma de distancias al centroide)', fontsize=11)
axes[0].set_title('Método del Codo', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette Score
k_opt = list(rango_k)[silhouettes.index(max(silhouettes))]
colores_sil = ['#1565C0' if k == k_opt else '#90CAF9' for k in rango_k]
bars = axes[1].bar(rango_k, silhouettes, color=colores_sil, edgecolor='white', width=0.6)
for bar, val in zip(bars, silhouettes):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.002,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title(f'Silhouette Score (mejor: K={k_opt})', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\n=== TABLA RESUMEN ===')
for k, inercia, sil in zip(rango_k, inercias, silhouettes):
    marca = ' ◄ mejor silhouette' if k == k_opt else ''
    print(f'  K={k}  Inercia: {inercia:8.1f}  Silhouette: {sil:.3f}{marca}')

In [ ]:
# Aplicar K-Means con K=4 (ajusta este valor según lo que viste en los gráficos)
K_ELEGIDO = 4  # 🔧 PARÁMETRO: cámbialo y observa cómo cambian los clusters

kmeans = KMeans(n_clusters=K_ELEGIDO, random_state=42, n_init=10)
kmeans.fit(X_scaled)

df['cluster_kmeans'] = kmeans.labels_

sil_final = silhouette_score(X_scaled, kmeans.labels_)
print(f'K elegido         : {K_ELEGIDO}')
print(f'Inercia final     : {kmeans.inertia_:.1f}')
print(f'Silhouette Score  : {sil_final:.3f}')

print('\n=== TAMAÑO DE CADA CLUSTER ===')
for cid, cnt in df['cluster_kmeans'].value_counts().sort_index().items():
    print(f'  Cluster {cid}: {cnt:3d} huéspedes ({cnt/len(df)*100:.1f}%)')

In [ ]:
# Perfil promedio de cada cluster
perfil = df.groupby('cluster_kmeans')[features_cluster].mean().round(2)

print('=== PERFIL PROMEDIO POR CLUSTER ===')
print(perfil.T.to_string())

print('\n=== CANAL MÁS FRECUENTE ===')
print(df.groupby('cluster_kmeans')['canal_reserva']
        .agg(lambda x: x.value_counts().index[0]))

print('\n=== PAÍS MÁS FRECUENTE ===')
print(df.groupby('cluster_kmeans')['pais_origen']
        .agg(lambda x: x.value_counts().index[0]))

In [ ]:
paleta  = ['#1976D2','#F57C00','#388E3C','#7B1FA2','#D32F2F','#0288D1']
colores = [paleta[i % len(paleta)] for i in range(K_ELEGIDO)]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Clusters en espacio PCA
for i in range(K_ELEGIDO):
    mask = df['cluster_kmeans'] == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colores[i], label=f'Cluster {i}',
                    alpha=0.65, s=55, edgecolors='white', linewidth=0.4)
axes[0].set_xlabel('PC1', fontsize=11)
axes[0].set_ylabel('PC2', fontsize=11)
axes[0].set_title('K-Means en espacio PCA', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Radar de perfiles (barras agrupadas)
metrics_radar = ['noches','precio_noche','solicitudes_especiales',
                 'reservas_previas','dias_anticipacion','valoracion_cliente']
perfil_norm = perfil[metrics_radar].copy()
for col in perfil_norm.columns:
    rng = perfil_norm[col].max() - perfil_norm[col].min()
    perfil_norm[col] = (perfil_norm[col] - perfil_norm[col].min()) / (rng if rng > 0 else 1)

x_pos = np.arange(len(metrics_radar))
bar_w = 0.8 / K_ELEGIDO
for i in range(K_ELEGIDO):
    axes[1].bar(x_pos + i * bar_w - 0.4 + bar_w/2,
                perfil_norm.iloc[i],
                width=bar_w, color=colores[i], label=f'Cluster {i}',
                alpha=0.85, edgecolor='white')

axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([m.replace('_',' ') for m in metrics_radar],
                         rotation=25, ha='right', fontsize=9)
axes[1].set_title('Perfil normalizado de cada cluster', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Valor normalizado (0-1)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 💬 Reflexión 3
> **Mira el perfil promedio de cada cluster y ponle un nombre de negocio a cada uno.** Ejemplos:
> *'Huésped corporativo de paso'*, *'Familia de vacaciones'*, *'Turista de lujo'*, *'Viajero espontáneo'*...
>
> | Cluster | Nombre propuesto | Característica principal |
> |---|---|---|
> | 0 | | |
> | 1 | | |
> | 2 | | |
> | 3 | | |
>
> **¿Qué acción de marketing haría el hotel para cada perfil?**
>
> *(Edita la tabla arriba)*

---

---
## PARTE 4 · Clustering Jerárquico 🟤
**⏱ 8 minutos**

El **clustering jerárquico** no requiere que definas K de antemano. Construye un árbol de fusiones (dendrograma) que puedes cortar a la altura que desees para obtener distintos números de clusters.

Es más lento que K-Means para datasets grandes, pero el dendrograma revela la **estructura natural** del dato de una forma muy visual.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

# Usamos una muestra de 80 puntos para que el dendrograma sea legible
np.random.seed(42)
idx_muestra = np.random.choice(len(X_scaled), size=80, replace=False)
X_muestra   = X_scaled[idx_muestra]

# Calcular la jerarquía
Z = linkage(X_muestra, method='ward')

fig, ax = plt.subplots(figsize=(18, 6))
dendrogram(Z, ax=ax, color_threshold=15,
           leaf_rotation=90, leaf_font_size=7,
           above_threshold_color='#90A4AE')
ax.axhline(y=15, color='#D32F2F', linestyle='--', linewidth=1.5, label='Corte propuesto')
ax.set_title('Dendrograma — Clustering Jerárquico (Ward, muestra 80 reservas)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Índice de reserva (muestra)', fontsize=11)
ax.set_ylabel('Distancia de fusión', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print('Consejo: el número de grupos equivale a las líneas verticales que cruza la línea roja.')

In [ ]:
# Aplicar al dataset completo
K_HIER = 4  # 🔧 Ajusta según lo que viste en el dendrograma

hier = AgglomerativeClustering(n_clusters=K_HIER, linkage='ward')
df['cluster_hier'] = hier.fit_predict(X_scaled)

sil_hier = silhouette_score(X_scaled, df['cluster_hier'])
print(f'K jerárquico      : {K_HIER}')
print(f'Silhouette Score  : {sil_hier:.3f}')

print('\n=== TAMAÑO DE CLUSTERS (Jerárquico) ===')
for cid, cnt in df['cluster_hier'].value_counts().sort_index().items():
    print(f'  Cluster {cid}: {cnt:3d} ({cnt/len(df)*100:.1f}%)')

# Comparar con K-Means
print('\n=== COMPARACIÓN K-MEANS vs JERÁRQUICO ===')
print(f'  Silhouette K-Means     : {sil_final:.3f}')
print(f'  Silhouette Jerárquico  : {sil_hier:.3f}')
mejor = 'K-Means' if sil_final >= sil_hier else 'Jerárquico'
print(f'  Mejor separación       : {mejor}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# K-Means
for i in range(K_ELEGIDO):
    mask = df['cluster_kmeans'] == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=paleta[i % len(paleta)], label=f'C{i}',
                    alpha=0.6, s=45, edgecolors='white', linewidth=0.3)
axes[0].set_title(f'K-Means (K={K_ELEGIDO})  Silhouette: {sil_final:.3f}',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Jerárquico
for i in range(K_HIER):
    mask = df['cluster_hier'] == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=paleta[i % len(paleta)], label=f'C{i}',
                    alpha=0.6, s=45, edgecolors='white', linewidth=0.3)
axes[1].set_title(f'Jerárquico (K={K_HIER})  Silhouette: {sil_hier:.3f}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparación visual: K-Means vs Clustering Jerárquico',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 💬 Reflexión 4
> 1. **¿Los dos algoritmos producen grupos similares o muy diferentes? ¿Dónde difieren más?**
> 2. **¿Cuándo preferirías usar clustering jerárquico sobre K-Means?** Piensa en tamaño del dataset, necesidad de visualizar la jerarquía, etc.
>
> *(Escribe aquí)*

---

---
## PARTE 5 · DBSCAN: Detección de anomalías y reservas atípicas 🔴
**⏱ 10 minutos**

**DBSCAN** (Density-Based Spatial Clustering) agrupa puntos por densidad y **marca como ruido (−1) los puntos aislados**. Esos puntos aislados son nuestras **anomalías**: reservas con comportamiento inusual.

A diferencia de K-Means:
- No necesitas definir K de antemano
- Detecta clusters de forma irregular
- Identifica outliers automáticamente

### Parámetros clave
| Parámetro | Descripción | Efecto si sube |
|---|---|---|
| `eps` | Radio de vecindad | Menos clusters, más puntos agrupados |
| `min_samples` | Puntos mínimos para formar un cluster | Más ruido, clusters más densos |

In [ ]:
from sklearn.cluster import DBSCAN

# 🔧 PARÁMETROS: modifícalos y observa cómo cambia el número de anomalías
EPS         = 1.2
MIN_SAMPLES = 8

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
df['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

n_clusters  = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)
n_anomalias = (df['cluster_dbscan'] == -1).sum()

print(f'eps={EPS}  min_samples={MIN_SAMPLES}')
print(f'Clusters encontrados : {n_clusters}')
print(f'Anomalías (ruido=-1) : {n_anomalias} ({n_anomalias/len(df)*100:.1f}% del total)')

print('\n=== TAMAÑO DE CLUSTERS DBSCAN ===')
for cid, cnt in df['cluster_dbscan'].value_counts().sort_index().items():
    etiq = '⚠️  ANOMALÍA' if cid == -1 else f'  Cluster {cid}'
    print(f'  {etiq}: {cnt:3d} reservas ({cnt/len(df)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Clusters DBSCAN en PCA
labels_unicos = sorted(df['cluster_dbscan'].unique())
colores_db    = ['#D32F2F'] + paleta[:max(0, len(labels_unicos)-1)]  # rojo = anomalía

for label, color in zip(labels_unicos, colores_db):
    mask   = df['cluster_dbscan'] == label
    nombre = '⚠ Anomalía' if label == -1 else f'Cluster {label}'
    size   = 90 if label == -1 else 45
    marker = 'X' if label == -1 else 'o'
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=nombre, alpha=0.7,
                    s=size, marker=marker, edgecolors='white', linewidth=0.3)

axes[0].set_title(f'DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})\n'
                  f'{n_anomalias} anomalías detectadas',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Perfil de anomalías vs normales
df['es_anomalia'] = df['cluster_dbscan'] == -1
comparacion = df.groupby('es_anomalia')[['precio_noche','noches',
                                          'precio_total','dias_anticipacion']].mean().T
comparacion.columns = ['Normal','Anomalía']

x_comp = np.arange(len(comparacion))
axes[1].bar(x_comp - 0.2, comparacion['Normal'],   width=0.35,
            label='Normal', color='#1976D2', alpha=0.85, edgecolor='white')
axes[1].bar(x_comp + 0.2, comparacion['Anomalía'], width=0.35,
            label='Anomalía', color='#D32F2F', alpha=0.85, edgecolor='white')
axes[1].set_xticks(x_comp)
axes[1].set_xticklabels(comparacion.index, rotation=15, ha='right')
axes[1].set_title('Perfil promedio: Normal vs Anomalía', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Inspeccionar las anomalías detectadas
anomalias = df[df['cluster_dbscan'] == -1][['id','tipo_hotel','canal_reserva',
                                              'noches','precio_noche','precio_total',
                                              'dias_anticipacion','pais_origen']]
print(f'=== RESERVAS DETECTADAS COMO ANOMALÍAS ({len(anomalias)}) ===')
print(anomalias.sort_values('precio_total', ascending=False).head(20).to_string(index=False))

### 💬 Reflexión 5
> 1. **¿Qué características tienen las reservas marcadas como anomalías?** (revisa la tabla y el gráfico de barras)
> 2. **¿Podría DBSCAN usarse en un hotel real?** Piensa en un caso concreto: detección de fraude, errores de carga de datos, reservas VIP inusuales...
> 3. **¿Qué pasa si subes `eps` a 2.0?** Ejecútalo y compara cuántas anomalías quedan.
>
> *(Escribe aquí)*

---

---
## PARTE 6 · Comparación final de los tres algoritmos
**⏱ 7 minutos**

Ahora ponemos en perspectiva los tres métodos que aplicaste sobre el mismo dataset.

In [ ]:
print('=' * 62)
print('       COMPARACIÓN FINAL DE ALGORITMOS NO SUPERVISADOS')
print('=' * 62)

print(f'\n🔵 K-MEANS  (K={K_ELEGIDO})')
print(f'   Silhouette Score : {sil_final:.3f}')
print(f'   Inercia          : {kmeans.inertia_:.1f}')
for cid, cnt in df['cluster_kmeans'].value_counts().sort_index().items():
    print(f'   Cluster {cid}        : {cnt:3d} huéspedes ({cnt/len(df)*100:.1f}%)')

print(f'\n🟤 JERÁRQUICO  (Ward, K={K_HIER})')
print(f'   Silhouette Score : {sil_hier:.3f}')
for cid, cnt in df['cluster_hier'].value_counts().sort_index().items():
    print(f'   Cluster {cid}        : {cnt:3d} huéspedes ({cnt/len(df)*100:.1f}%)')

print(f'\n🔴 DBSCAN  (eps={EPS}, min_samples={MIN_SAMPLES})')
print(f'   Clusters normales: {n_clusters}')
print(f'   Anomalías        : {n_anomalias} ({n_anomalias/len(df)*100:.1f}%)')
print()

tabla = pd.DataFrame({
    'Característica': [
        '¿Define K?', 'Detecta outliers', 'Forma de clusters',
        'Escalable (big data)', 'Interpretabilidad', 'Métrica principal'
    ],
    'K-Means': [
        'Sí (manual)', 'No', 'Esféricas',
        'Alta', 'Alta (centroides)', f'Silhouette: {sil_final:.3f}'
    ],
    'Jerárquico': [
        'Después (dendrograma)', 'No', 'Cualquier forma',
        'Baja (O(n²))', 'Muy alta (dendrograma)', f'Silhouette: {sil_hier:.3f}'
    ],
    'DBSCAN': [
        'No', 'Sí (ruido=-1)', 'Cualquier forma',
        'Media', 'Media (eps/min_samples)', f'{n_anomalias} anomalías'
    ]
})
print(tabla.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titulos = [
    f'🔵 K-Means (K={K_ELEGIDO})\nSilhouette: {sil_final:.3f}',
    f'🟤 Jerárquico (K={K_HIER})\nSilhouette: {sil_hier:.3f}',
    f'🔴 DBSCAN\n{n_anomalias} anomalías detectadas'
]
cluster_cols = ['cluster_kmeans', 'cluster_hier', 'cluster_dbscan']

for ax, col, titulo in zip(axes, cluster_cols, titulos):
    labels_u = sorted(df[col].unique())
    colors_u = ['#D32F2F'] + paleta if -1 in labels_u else paleta
    for label, color in zip(labels_u, colors_u):
        mask   = df[col] == label
        nombre = '⚠ Anomalía' if label == -1 else f'C{label}'
        size   = 80 if label == -1 else 40
        marker = 'X' if label == -1 else 'o'
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=color, label=nombre, alpha=0.65,
                   s=size, marker=marker, edgecolors='white', linewidth=0.3)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=8, loc='best'); ax.grid(True, alpha=0.3)

plt.suptitle('Comparación visual de los tres algoritmos (espacio PCA)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparacion_algoritmos.png', dpi=130, bbox_inches='tight')
plt.show()
print('📊 Gráfico guardado como comparacion_algoritmos.png')

---
## 💬 Reflexión Final

> **1. ¿Para qué decisión de negocio en un hotel usarías K-Means? ¿Y DBSCAN?**
>
> *(Escribe aquí)*

> **2. Si el director del hotel te dice: "quiero saber si alguna reserva es sospechosa de fraude", ¿qué algoritmo de los tres elegirías y por qué?**
>
> *(Escribe aquí)*

> **3. PCA redujo 10 dimensiones a 2. ¿Qué información se pierde en ese proceso? ¿Importa?**
>
> *(Escribe aquí)*

> **4. ¿Cuál es la diferencia fundamental entre aprendizaje supervisado y no supervisado en términos de lo que el modelo "conoce" antes de aprender?**
>
> *(Escribe aquí)*

---

## ✅ ¡Actividad completada!

| | Lo que hiciste hoy |
|---|---|
| 🟣 PCA | Redujiste 10 dimensiones a 2 para visualizar patrones ocultos |
| 🔵 K-Means | Segmentaste huéspedes en perfiles de negocio usando el método del codo y Silhouette Score |
| 🟤 Jerárquico | Exploraste la estructura natural del dato con un dendrograma sin definir K |
| 🔴 DBSCAN | Detectaste reservas atípicas automáticamente sin necesitar etiquetas |

**Los tres algoritmos son no supervisados**: ninguno usó la respuesta correcta. Cada uno revela una perspectiva diferente del mismo dato.

---
## 🎯 RETO EXTRA (opcional)

Elige **una** de estas extensiones:

**A. Optimizar DBSCAN automáticamente**
Escribe un bucle que pruebe distintos valores de `eps` (0.5, 0.8, 1.0, 1.2, 1.5, 2.0) y `min_samples` (5, 8, 10) e imprima cuántas anomalías y clusters produce cada combinación. ¿Qué combinación da el número más razonable de anomalías?

**B. Añadir una variable nueva al clustering**
El dataset tiene `tipo_hotel` y `canal_reserva` como variables categóricas. Usa `pd.get_dummies()` para convertirlas a numéricas, incorpóralas a `features_cluster` y vuelve a ejecutar K-Means. ¿Cambian los clusters?

**C. Visualización 3D con PCA**
Calcula 3 componentes principales en lugar de 2 y visualiza los clusters de K-Means en 3D con `mpl_toolkits.mplot3d`. ¿Se separan mejor los grupos?

```python
# Pista para el reto C:
from mpl_toolkits.mplot3d import Axes3D
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_scaled)
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')
# ... tu código aquí
```